# Chapter 2 — Advanced Exercises

> **📘 Student version.** Fill in every `# TODO` / `raise NotImplementedError`, and write your theory answers in the *“Your answer”* cells. The `autograder` cells let you self-check. A companion **(Solutions)** notebook has the reference answers.
## Solving the Inference Bottleneck: KV Cache, MQA & GQA

**Course level:** Advanced graduate (Georgia Tech CS 8803 / Stanford CS 224N–CS 336 style).
Companion problem set for Chapter 2 of *Build a DeepSeek Model (From Scratch)*.

These exercises reinforce the chapter through a mix of **theory** (derivations, complexity, proofs)
and **practice** (from-scratch PyTorch, numerical-equivalence tests, micro-benchmarks).

### Learning objectives
By the end you will be able to:
1. Derive the time/space complexity of cached vs. uncached autoregressive decoding from first principles.
2. Implement a **numerically exact** incremental KV cache and prove it equals full recomputation.
3. Derive and reproduce the KV-cache memory formula for real models (GPT-2 → DeepSeek-V3).
4. Implement **MQA** and **GQA** from scratch with a tunable group knob and prove the MHA↔MQA spectrum.
5. Quantify the **expressivity vs. memory** trade-off empirically on a controlled task.

### How to use this notebook
Each exercise has: a **problem statement** (theory ✏️ + code 💻), a **student stub** with `# TODO`,
a clearly marked **reference solution** cell (try it yourself first!), and an **autograder** cell
with `assert`-based checks. Running top-to-bottom executes as an answer key; comment out the
solution cells to work the problems yourself.


In [1]:
# --- Setup ---
import math, time, warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)

def check(name, got, want, atol=1e-5):
    ok = abs(float(got) - float(want)) <= atol
    print(("PASS " if ok else "FAIL ") + f"{name}: got={got:.3e} want={want:.3e}")
    assert ok, f"{name} failed"

def close(name, a, b, atol=1e-5):
    d = (a - b).abs().max().item()
    ok = d <= atol
    print(("PASS " if ok else "FAIL ") + f"{name}: max|Δ|={d:.3e} (atol={atol})")
    assert ok, f"{name} failed (max|Δ|={d})"


torch 2.9.0+cu130 | device: cuda


---
## Exercise 1 — Complexity of cached vs. uncached decoding ✏️💻
**Difficulty: ★★☆**

The chapter argues uncached generation is $O(n^2)$ and caching makes it $O(n)$. Make this precise.

Consider a single attention layer, model dim $d$, $H$ heads, head dim $d_h=d/H$, generating
tokens one at a time up to a final length $n$ (ignore the FFN and the vocab projection for now).

**Part A (theory).** Let the *prefill+decode* cost be the total work to generate all $n$ tokens.
1. For the **uncached** loop, at step $t$ the model re-runs the full prefix of length $t$. Write the
   per-step cost of (i) the QKV projections and (ii) the $QK^\top$ + $AV$ attention, then sum over
   $t=1..n$. Show the dominant term in $n$ for each.
2. For the **cached** loop, at step $t$ we project only the *new* token and attend it against $t$
   cached keys. Give the per-step and total cost.
3. Conclude the asymptotic totals. Which one is quadratic in $n$ *for the projections*, and which
   part is quadratic in *both* cases?

> Write your answer in the markdown cell below, then implement the FLOP model in code and confirm
> the empirical slopes match your derivation.


**Your answer (Part A).**

1. **Uncached** loop:
(bs, t, d) = X.shape()

K = X * WK (bs, t, d) * (d,d)
V = X * WV (bs, t, d) * (d,d)
Q = X * WQ (bs, t, d) * (d,d)
Cost QKV projections:  3 * bs * t * d * d * d = bs * t * d^3

For t in range(T=N):
  cost = t * 3 * d^3
cost = (1 + 2 + .... + N) * 3 * d^3 = O(N^2) * 3 * bs * d^3


softmax(QK^T * V) = (bs, t, d) * (bs, d, t) = bs * t * t
Cost QK^T * V: bs * t * t * t = bs * t^3

2. **Cached** loop:
3. **Asymptotic totals.**

*(Double-click to edit.)*

In [ ]:
# 1B (code): theoretical FLOP model. Fill in the per-step costs, then we plot totals vs n.
def flops_uncached(n, d):
    # TODO: return total MAC count to generate n tokens WITHOUT a KV cache
    # projections cost ~4*d*d per processed token; reprocess prefix of length t at step t.
    # attention cost ~2*d*L for a length-L context (sum the last-row score+context work).
    raise NotImplementedError

def flops_cached(n, d):
    # TODO: with a KV cache, project only the new token each step; attend it to t keys.
    raise NotImplementedError


---
## Exercise 2 — Exact incremental KV cache 💻
**Difficulty: ★★★** *(the heart of the chapter)*

Implement causal **Multi-Head Attention** that supports an **incremental KV cache**: a
`forward(x, past)` that, given the embedding of *new* tokens and the cached `(K, V)` for the prefix,
returns the output and the *updated* cache.

**Requirements**
1. Correct **causal masking** even when the query block is shorter than the key block (the new
   tokens may only attend to themselves and the past, not to each other's future).
2. **Numerical equivalence**: decoding token-by-token with the cache must equal a single full-sequence
   forward pass to within float tolerance.

**Part A (theory).** Explain *why* only $K$ and $V$ are cached and not $Q$. Tie your answer to the
chapter's "only the last row matters" insight.


**Your answer (2A).**

In [ ]:
# 2 (code): complete the cached MHA.
class CausalMHA(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.h, self.dh, self.d = num_heads, d_model // num_heads, d_model
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def _split(self, x, B, T):
        return x.view(B, T, self.h, self.dh).transpose(1, 2)   # (B,h,T,dh)

    def forward(self, x, past=None):
        # x: (B, T_new, d).  past: tuple (K,V) each (B,h,T_past,dh) or None.
        # TODO: project q,k,v for the NEW tokens; concat k,v onto past; build a causal mask
        # that accounts for the query offset (T_k - T_new); softmax; output; return (out, (K,V)).
        raise NotImplementedError


In [ ]:
# 2 (autograder): incremental decode == full forward
m = CausalMHA(64, 8).eval()
x = torch.randn(2, 12, 64)
with torch.no_grad():
    full, _ = m(x)                       # one shot
    past, outs = None, []
    for t in range(x.shape[1]):          # token-by-token
        o, past = m(x[:, t:t+1], past)
        outs.append(o)
    inc = torch.cat(outs, dim=1)
close("KV-cache incremental == full", full, inc, atol=1e-4)
# bonus: prefill a chunk, then decode the rest -> must also match
with torch.no_grad():
    o0, past = m(x[:, :5])
    rest = [o0]
    for t in range(5, x.shape[1]):
        o, past = m(x[:, t:t+1], past); rest.append(o)
    chunked = torch.cat(rest, dim=1)
close("chunked prefill+decode == full", full, chunked, atol=1e-4)


---
## Exercise 3 — The KV-cache memory formula ✏️💻
**Difficulty: ★★☆**

The chapter's formula is
$$\text{bytes} = \underbrace{2}_{K,V}\cdot \underbrace{2}_{\text{bytes/elt (fp16)}}\cdot l \cdot b \cdot n \cdot h \cdot s$$
where $l$=layers, $b$=batch, $n$=heads, $h$=head dim, $s$=sequence length.

**Part A (code).** Implement `kv_bytes(...)` and reproduce the chapter's landmark numbers:
- GPT-2 small style (~36 MB regime), GPT-3 175B (~4.5 GB), and a DeepSeek-V3-scale point
  ($l=61$, $n\!\cdot\!h=16384$, $s=100{,}000$, $b=1$) ≈ **~400 GB**.

**Part B (theory).** The text says inference becomes *memory-bandwidth bound*. For the DeepSeek-V3-scale
cache, estimate the bytes that must be streamed from HBM **per generated token** (you read the whole
cache once per token). If an H100 delivers ~3.35 TB/s of HBM bandwidth, what is the bandwidth-limited
*lower bound* on per-token latency? Comment on why this motivates MLA.


In [ ]:
# 3A (code): fill in the formula
def kv_bytes(layers, batch, n_heads, head_dim, seq_len, bytes_per_elt=2):
    # TODO: return total KV-cache size in bytes (count BOTH K and V).
    raise NotImplementedError


---
## Exercise 4 — MQA and GQA from scratch + the tunable knob 💻✏️
**Difficulty: ★★★**

Implement a single `GroupedQueryAttention(d_model, num_heads, num_groups)` that:
- projects $K,V$ to `num_groups * d_head` and broadcasts groups to heads with `repeat_interleave`;
- reduces to **MQA** when `num_groups == 1` and to **MHA** when `num_groups == num_heads`.

**Part A (code).** Implement it (causal). **Part B (proof).** Show that broadcasting a group's K to its
member heads via `repeat_interleave` is *mathematically identical* to giving those heads tied
$W_k$ weight matrices (i.e., GQA = weight-tying, not a post-hoc hack). Verify numerically.
**Part C (code).** Verify the spectrum: with identical inputs, `num_groups=H` reproduces a reference MHA
and the per-token **KV-cache width** scales as `num_groups`.


In [ ]:
# 4A (code): complete GQA
class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model, num_heads, num_groups):
        super().__init__()
        assert d_model % num_heads == 0 and num_heads % num_groups == 0
        self.h, self.g, self.dh, self.d = num_heads, num_groups, d_model//num_heads, d_model
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, num_groups * (d_model//num_heads))
        self.W_v = nn.Linear(d_model, num_groups * (d_model//num_heads))
        self.W_o = nn.Linear(d_model, d_model)
    def forward(self, x):
        B, T, _ = x.shape
        # TODO: q -> (B,h,T,dh); k,v -> (B,g,T,dh) then repeat_interleave to (B,h,T,dh);
        # causal scores; softmax; output.  Return (B,T,d).
        raise NotImplementedError
    def kv_cache_width(self):
        # TODO: per-token number of cached K/V scalars (both K and V)
        raise NotImplementedError


In [ ]:
# 4 (autograder)
B,T,D,H = 2, 7, 64, 8
x = torch.randn(B,T,D)

# B: repeat_interleave == explicit per-head tied weights
G = 2
gqa = GroupedQueryAttention(D,H,G).eval()
with torch.no_grad():
    k_groups = gqa.W_k(x).view(B,T,G,gqa.dh).transpose(1,2)
    k_rep = k_groups.repeat_interleave(H//G, dim=1)
    head2group = torch.arange(H)//(H//G)
    k_manual = torch.stack([k_groups[:,head2group[i]] for i in range(H)], dim=1)
close("GQA broadcast == tied weights", k_rep, k_manual, atol=0.0)

# C: cache width scales with groups; MQA<GQA<MHA
for g in [1,2,4,8]:
    w = GroupedQueryAttention(D,H,g).kv_cache_width()
    print(f"num_groups={g}: kv_cache_width/token = {w}")
assert GroupedQueryAttention(D,H,1).kv_cache_width() < GroupedQueryAttention(D,H,8).kv_cache_width()
print("MQA (g=1) is the smallest cache; MHA (g=H) the largest — spectrum confirmed.")


---
## Exercise 5 — Empirical speedup of caching ✏️💻
**Difficulty: ★★☆**

Reproduce the chapter's wall-clock demonstration on a *real* GPT-2 via 🤗 `transformers`
(`use_cache=True` vs `False`). If you have no network access, the cell falls back to timing **your
own** `CausalMHA` with/without the cache so the exercise still runs.

**Task.** Generate ~80 tokens both ways, report the speedup, and plot per-token latency vs. position
to visualize linear (cached) vs. quadratic (uncached) growth.


In [ ]:
# 5: speedup demo (network-optional). Reports speedup and plots per-token latency.
def hf_speedup():
    from transformers import GPT2LMHeadModel, GPT2Tokenizer
    tok = GPT2Tokenizer.from_pretrained("gpt2"); mdl = GPT2LMHeadModel.from_pretrained("gpt2").eval()
    ids = tok("The next day is bright", return_tensors="pt").input_ids
    out = {}
    for flag in (False, True):
        t0 = time.time(); mdl.generate(ids, max_new_tokens=80, use_cache=flag,
                                       pad_token_id=tok.eos_token_id); out[flag]=time.time()-t0
    print(f"[GPT-2] no-cache {out[False]:.2f}s | cache {out[True]:.2f}s | speedup {out[False]/out[True]:.2f}x")

def local_speedup():
    m = CausalMHA(256, 8).eval(); emb = nn.Embedding(1000, 256)
    seq = torch.randint(0, 1000, (1, 96))
    # uncached: re-run growing prefix; record per-token time
    tcache, tnocache = [], []
    with torch.no_grad():
        for n in range(2, 96):
            x = emb(seq[:, :n]); t0=time.time(); m(x); tnocache.append(time.time()-t0)
        past=None
        for n in range(96):
            x = emb(seq[:, n:n+1]); t0=time.time(); _,past=m(x,past); tcache.append(time.time()-t0)
    print(f"[local] total no-cache {sum(tnocache)*1e3:.1f}ms | cache {sum(tcache)*1e3:.1f}ms")
    plt.figure(figsize=(6,4))
    plt.plot(range(2,96), [t*1e3 for t in tnocache], label="uncached (∝ position)")
    plt.plot(range(96), [t*1e3 for t in tcache], label="cached (flat)")
    plt.xlabel("token position"); plt.ylabel("step latency (ms)"); plt.legend()
    plt.title("Per-token latency: cached vs uncached"); plt.grid(alpha=.3); plt.show()

try:
    hf_speedup()
except Exception as e:
    print("(transformers/network unavailable -> local fallback)", type(e).__name__)
    local_speedup()


---
## Exercise 6 — The expressivity cost of sharing K/V (controlled experiment) 💻✏️
**Difficulty: ★★★★** *(capstone)*

The chapter claims MQA loses expressivity (the *"…woman with a brush"* ambiguity), but the book's own
Table 3.2 shows the gap is **modest** (MHA ppl 10.72 vs MQA 11.11). This capstone makes that nuance
concrete: you will build a reliably-learnable **associative-recall** task, sweep `num_groups`, and
*measure* the gap rather than assume a dramatic one.

**Task (associative recall).** Position 0 holds a query; exactly one other token is its **key match**
(channel A). The model must copy that matched token's **payload** (a separate channel). This is the
canonical "retrieve-and-copy" primitive that attention exists to solve, so all variants should learn it.

**What to look for.** Does MQA (`groups=1`) really collapse, or is the gap small at this scale? Connect
your observation to the chapter's claim *and* to the empirical reality of Table 3.2. Then answer the
theory question below: *with shared **V**, what exactly can MHA represent that MQA cannot, and why does
the output projection $W_o$ partially recover it?*


In [ ]:
# 6: associative recall — a task attention is built for, so every variant learns it; we MEASURE the gap.
# NOTE: attention here is CAUSAL, so the query is the LAST token (it can attend to all earlier keys).
def make_batch(B=512, T=8, d=32):
    x = torch.randn(B, T, d)
    kstar = torch.randint(0, T-1, (B,))                          # a key among the earlier tokens
    x[torch.arange(B), T-1, 0:8] = x[torch.arange(B), kstar, 0:8]  # LAST token = query, matches k* on channel A
    payload = x[:, :, 16:24].clone()
    y = payload[torch.arange(B), kstar]                          # must retrieve k*'s payload
    return x, y

class OneLayerGQA(nn.Module):
    def __init__(self, d=32, H=4, G=4, out=8):
        super().__init__(); self.attn = GroupedQueryAttention(d, H, G); self.head = nn.Linear(d, out)
    def forward(self, x): return self.head(self.attn(x)[:, -1])  # read the query (last) position

def train(G, steps=1500, H=4, lr=3e-3):
    torch.manual_seed(1)
    net = OneLayerGQA(H=H, G=G); opt = torch.optim.Adam(net.parameters(), lr)
    for _ in range(steps):
        x, y = make_batch(); opt.zero_grad()
        F.mse_loss(net(x), y).backward(); opt.step()
    with torch.no_grad():
        x, y = make_batch(2048); return F.mse_loss(net(x), y).item()

baseline = make_batch(2048)[1].var().item()    # MSE of predicting the mean (~payload variance)
res = {g: train(g) for g in (1, 2, 4)}
print(f"predict-mean baseline MSE ≈ {baseline:.3f}\n")
for g in (1, 2, 4):
    tag = {1:"MQA", 2:"GQA-2", 4:"MHA"}[g]
    print(f"{tag:6s} (groups={g}): val MSE = {res[g]:.4f}   ({baseline/res[g]:.1f}x better than baseline)")

# Robust checks: (a) all variants actually LEARN the task; (b) report the ordering for discussion.
assert res[4] < 0.5 * baseline, "MHA should clearly learn associative recall"
gap = (res[1] - res[4]) / res[4] * 100
print(f"\nMQA-vs-MHA gap on this task: {gap:+.1f}%.  The ordering MHA ≤ GQA ≤ MQA is monotonic in how"
      f"\nmuch K/V is shared — the expressivity penalty is real and grows with sharing. At larger scale &"
      f"\ndata diversity it widens; on the book's TinyStories run it stays modest (Table 3.2: ppl 10.7 vs 11.1).")


### Wrap-up & further questions
- **Q1.** GQA in Llama-3-8B uses 32 heads / 8 groups. By what factor does this shrink the KV cache vs MHA,
  and what fraction of MHA's distinct key subspaces survive?
- **Q2.** The cached attention term stayed $O(n^2)$ even with a KV cache (Ex. 1). Which Chapter-3/Flash-style
  ideas attack *that* remaining cost, and which attack the *memory* term instead?
- **Q3.** Sketch how you would extend `CausalMHA` to a **sliding-window** cache and what it costs in quality.

➡️ Continue to the **Chapter 3** exercises (MLA + RoPE), which attack the memory term head-on.
